In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.optimize import curve_fit
from scipy.interpolate import PchipInterpolator 
import warnings

from astropy.visualization import ImageNormalize, AsymmetricPercentileInterval, LinearStretch

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def clean_spectral_spikes(flux, window_size=7, sigma_thresh=4.5):
    """
    Removes narrow pixel spikes (hot pixels/cosmic rays) using a rolling median baseline.
    """
    n = len(flux)
    half_w = window_size // 2
    rolling_median = np.zeros_like(flux)
    
    for i in range(n):
        start = max(0, i - half_w)
        end = min(n, i + half_w + 1)
        rolling_median[i] = np.nanmedian(flux[start:end])
        
    residual = flux - rolling_median
    global_mad = np.nanmedian(np.abs(residual - np.nanmedian(residual)))
    robust_std = 1.4826 * global_mad if global_mad > 0 else 1e-6
    
    spike_mask = residual > (sigma_thresh * robust_std)
    
    cleaned_flux = flux.copy()
    if np.any(spike_mask):
        cleaned_flux[spike_mask] = np.nan
        valid_indices = np.where(~np.isnan(cleaned_flux))[0]
        nan_indices = np.where(spike_mask)[0]
        
        if len(valid_indices) > 1:
            cleaned_flux[nan_indices] = np.interp(nan_indices, valid_indices, cleaned_flux[valid_indices])
            
    return cleaned_flux

def gaussian(x, amp, cen, wid):
    return amp * np.exp(-(x - cen)**2 / (2 * wid**2))

def double_gaussian(x, amp1, cen1, wid1, amp2, cen2, wid2):
    return gaussian(x, amp1, cen1, wid1) + gaussian(x, amp2, cen2, wid2)

def asymmetric_gaussian(x, amp, cen, wid_left, wid_right):
    y = np.zeros_like(x)
    left_mask = x <= cen
    right_mask = x > cen
    
    if np.any(left_mask):
        y[left_mask] = amp * np.exp(-(x[left_mask] - cen)**2 / (2 * wid_left**2))
    if np.any(right_mask):
        y[right_mask] = amp * np.exp(-(x[right_mask] - cen)**2 / (2 * wid_right**2))
        
    return y

def fit_pchip_continuum(waves, flux, anchor_ranges):
    anchor_waves = []
    anchor_fluxes = []

    for wmin, wmax in anchor_ranges:
        mask = (waves >= wmin) & (waves <= wmax) & ~np.isnan(flux) & (flux > 0)
        if np.sum(mask) > 0:  
            anchor_waves.append(np.median(waves[mask]))
            anchor_fluxes.append(np.percentile(flux[mask], 75))
            
    if len(anchor_waves) < 2: 
        return np.full_like(waves, np.nan)
        
    sort_idx = np.argsort(anchor_waves)
    anchor_waves = np.array(anchor_waves)[sort_idx]
    anchor_fluxes = np.array(anchor_fluxes)[sort_idx]
    
    try:
        pchip = PchipInterpolator(anchor_waves, anchor_fluxes, extrapolate=True)
        return pchip(waves)
    except Exception:
        return np.full_like(waves, np.nan)

def fit_single_ice(tau_array, wave_array, w_min, w_max, p0_tau, p0_wid, fixed_cen, wid_bounds):
    mask = (wave_array >= w_min) & (wave_array <= w_max) & ~np.isnan(tau_array)
    if np.sum(mask) < 3:
        return 0.0, fixed_cen, p0_wid, 0.0
    
    def bounded_gaussian(x, amp, wid):
        return gaussian(x, amp, fixed_cen, wid)
        
    try:
        popt, pcov = curve_fit(
            bounded_gaussian, wave_array[mask], tau_array[mask],
            p0=[p0_tau, p0_wid], 
            bounds=([0.0, wid_bounds[0]], [10.0, wid_bounds[1]])
        )
        err = np.sqrt(pcov[0,0]) if not np.isinf(pcov).all() else 0.0
        return popt[0], fixed_cen, popt[1], err
    except Exception:
        return 0.0, fixed_cen, p0_wid, 0.0

def fit_h2o_asymmetric(tau_array, wave_array, w_min, w_max, p0_tau, fixed_cen):
    mask = (wave_array >= w_min) & (wave_array <= w_max) & ~np.isnan(tau_array)
    if np.sum(mask) < 5:
        return 0.0, fixed_cen, 0.1, 0.2, 0.0
        
    def model(x, amp, wid_left, wid_right):
        return asymmetric_gaussian(x, amp, fixed_cen, wid_left, wid_right)
        
    try:
        popt, pcov = curve_fit(
            model, wave_array[mask], tau_array[mask],
            p0=[p0_tau, 0.15, 0.35],
            bounds=([0.0, 0.05, 0.10], [10.0, 0.30, 0.80]) 
        )
        err = np.sqrt(pcov[0,0]) if not np.isinf(pcov).all() else 0.0
        return popt[0], fixed_cen, popt[1], popt[2], err
    except Exception:
        return 0.0, fixed_cen, 0.15, 0.35, 0.0

def evaluate_and_fit_pah_features(waves, flux, subtract_pah=True, fit_34_feature=True, thresh_33=3.0, thresh_34=3.0):
    """
    Evaluates residual emission against local noise thresholds and conditionally fits
    3.3 um and/or 3.4 um features only when signal-to-noise requirements are met.
    """
    a_pah, c_pah, w_pah = 0.0, 3.29, 0.04
    a_34, c_34, w_34 = 0.0, 3.40, 0.03
    
    if not subtract_pah:
        return a_pah, c_pah, w_pah, a_34, c_34, w_34

    pah_mask = (waves >= 3.15) & (waves <= 3.55) & ~np.isnan(flux)
    if np.sum(pah_mask) < 5:
        return a_pah, c_pah, w_pah, a_34, c_34, w_34

    lw, lf = waves[pah_mask], flux[pah_mask]
    baseline = np.interp(lw, [lw[0], lw[-1]], [lf[0], lf[-1]])
    pah_only_flux = lf - baseline

    # Local noise estimate from off-peak continuum bands
    noise_mask = ((lw >= 3.15) & (lw <= 3.22)) | ((lw >= 3.50) & (lw <= 3.55))
    local_noise = np.nanstd(pah_only_flux[noise_mask]) if np.sum(noise_mask) > 2 else 1e-5
    if local_noise <= 0 or np.isnan(local_noise):
        local_noise = 1e-5

    # Check 3.3 um peak (3.25 - 3.35 um)
    m33 = (lw >= 3.25) & (lw <= 3.35)
    peak_33 = np.nanmax(pah_only_flux[m33]) if np.any(m33) else 0.0
    fit_33_needed = peak_33 > (thresh_33 * local_noise)

    # Check 3.4 um peak (3.38 - 3.48 um)
    m34 = (lw >= 3.38) & (lw <= 3.48)
    peak_34 = np.nanmax(pah_only_flux[m34]) if np.any(m34) else 0.0
    fit_34_needed = fit_34_feature and (peak_34 > (thresh_34 * local_noise))

    # Conditional Fitting Decision Tree
    if fit_33_needed and fit_34_needed:
        try:
            popt_2g, _ = curve_fit(
                double_gaussian, lw, pah_only_flux,
                p0=[peak_33, 3.29, 0.04, peak_34, 3.40, 0.03],
                bounds=([0, 3.25, 0.02, 0, 3.37, 0.015], 
                        [np.nanmax(flux) * 3, 3.35, 0.15, np.nanmax(flux) * 2, 3.46, 0.08])
            )
            a_pah, c_pah, w_pah, a_34, c_34, w_34 = popt_2g
        except Exception:
            pass
    elif fit_33_needed:
        try:
            popt_33, _ = curve_fit(
                gaussian, lw, pah_only_flux,
                p0=[peak_33, 3.29, 0.04],
                bounds=([0, 3.25, 0.02], [np.nanmax(flux) * 3, 3.35, 0.15])
            )
            a_pah, c_pah, w_pah = popt_33
        except Exception:
            pass
    elif fit_34_needed:
        try:
            popt_34, _ = curve_fit(
                gaussian, lw, pah_only_flux,
                p0=[peak_34, 3.40, 0.03],
                bounds=([0, 3.37, 0.015], [np.nanmax(flux) * 2, 3.46, 0.08])
            )
            a_34, c_34, w_34 = popt_34
        except Exception:
            pass

    return a_pah, c_pah, w_pah, a_34, c_34, w_34

# ==========================================
# MAIN PROCESSING FUNCTION
# ==========================================

def process_datacube(cube_path, output_fits="ice_maps.fits", 
                     output_png_base="ice_maps", output_diag="continuum_diagnostics.pdf",
                     subtract_pah=True, fit_34_feature=True, 
                     thresh_33=3.0, thresh_34=3.0, 
                     bin_factor=1, 
                     tau_thresholds={'H2O': 0.05, 'CO2': 0.01, 'CO': 0.01}): 
    
    print(f"Loading datacube: {cube_path}")
    
    with fits.open(cube_path) as hdul:
        data = hdul[0].data  
        header = hdul[0].header
        
        if 'WAVELENGTHS' in hdul:
            waves = hdul['WAVELENGTHS'].data
        else:
            n_waves = header['NAXIS3']
            waves = header['CRVAL3'] + (np.arange(1, n_waves + 1) - header['CRPIX3']) * header['CDELT3']

    flux_unit = header.get('BUNIT', 'MJy/sr')

    print("Trimming data to 1.4 - 5.0 um...")
    wave_mask = (waves >= 1.4) & (waves <= 5.0)
    waves = waves[wave_mask]
    data = data[wave_mask, :, :]
    
    nz, ny, nx = data.shape
    
    if bin_factor > 1:
        print(f"Binning spatial pixels by {bin_factor}x{bin_factor}...")
        new_ny = ny // bin_factor
        new_nx = nx // bin_factor
        data = data[:, :new_ny * bin_factor, :new_nx * bin_factor]
        data = data.reshape(nz, new_ny, bin_factor, new_nx, bin_factor).mean(axis=(2, 4))
        ny, nx = new_ny, new_nx
        if 'CRPIX1' in header: header['CRPIX1'] /= bin_factor
        if 'CRPIX2' in header: header['CRPIX2'] /= bin_factor
        if 'CDELT1' in header: header['CDELT1'] *= bin_factor
        if 'CDELT2' in header: header['CDELT2'] *= bin_factor

    ices = {
        'H2O': (2.6, 4, 2.0e-16),
        'CO2': (4.1, 4.4, 7.6e-17),
        'CO':  (4.45, 4.8, 1.1e-17)
    }
    
    anchor_ranges = [(1.4, 1.6), (2.0, 2.1), (2.45, 2.55), (3.65, 3.75), (4.05, 4.15), (4.4, 4.5), (4.8, 4.95)]
    
    maps = {ice: np.full((ny, nx), np.nan) for ice in ices}
    maps_err = {ice: np.full((ny, nx), np.nan) for ice in ices}
    maps_tau_peak = {ice: np.full((ny, nx), np.nan) for ice in ices} 
    
    maps['PAH_3.3'] = np.full((ny, nx), np.nan)
    maps['PAH_3.4'] = np.full((ny, nx), np.nan)
    maps['Flux_Ratio'] = np.full((ny, nx), np.nan) 
        
    continuum_cube = np.full_like(data, np.nan)
    tau_raw_cube = np.full_like(data, np.nan)
    tau_corr_cube = np.full_like(data, np.nan)
    
    print("Calculating median spectrum for background reference...")
    median_spectrum = np.nanmedian(data, axis=(1, 2))
    med_cont_150 = np.nan
    
    clean_med_spec = clean_spectral_spikes(median_spectrum, window_size=7, sigma_thresh=4.5)
    med_cont_spec = fit_pchip_continuum(waves, clean_med_spec, anchor_ranges)
    
    if not np.all(np.isnan(med_cont_spec)):
        med_cont_150 = np.interp(1.50, waves, med_cont_spec)

    nu = 10000.0 / waves
    sort_idx = np.argsort(nu)
    nu_sorted = nu[sort_idx]
    
    fit_ranges = {
        'H2O': (2.60, 4), 
        'CO2': (4.10, 4.4),
        'CO':  (4.55, 4.75)
    }

    print(f"Processing {ny}x{nx} spatial pixels...")
    
    for y in range(ny):
        for x in range(nx):
            flux = data[:, y, x]
            
            if np.sum(~np.isnan(flux) & (flux > 0)) < 5:
                continue
            
            clean_flux = clean_spectral_spikes(flux, window_size=7, sigma_thresh=4.5)
                
            cont_spec = fit_pchip_continuum(waves, clean_flux, anchor_ranges)
            if np.all(np.isnan(cont_spec)):
                continue
                
            continuum_cube[:, y, x] = cont_spec
            
            # CONDITIONAL FIT FOR 3.3 um AND 3.4 um FEATURES
            a_pah, c_pah, w_pah, a_34, c_34, w_34 = evaluate_and_fit_pah_features(
                waves, flux, subtract_pah=subtract_pah, fit_34_feature=fit_34_feature, 
                thresh_33=thresh_33, thresh_34=thresh_34
            )

            pah_33_array = gaussian(waves, a_pah, c_pah, w_pah) if a_pah > 0 else np.zeros_like(waves)
            pah_34_array = gaussian(waves, a_34, c_34, w_34) if a_34 > 0 else np.zeros_like(waves)
            total_pah_array = pah_33_array + pah_34_array
            
            flux_no_pah = flux - total_pah_array

            # CONVERT TO OPACITY (TAU) DOMAIN
            safe_flux = np.where(flux_no_pah > 0, flux_no_pah, np.nan)
            tau_raw = -np.log(safe_flux / cont_spec)
            
            tau_raw = np.nan_to_num(tau_raw, nan=0.0)
            tau_raw[tau_raw < 0] = 0.0
            tau_raw_cube[:, y, x] = tau_raw

            # FIT ICES
            t_h2o, c_h2o, wl_h2o, wr_h2o, err_h2o_tau = fit_h2o_asymmetric(
                tau_raw, waves, fit_ranges['H2O'][0], fit_ranges['H2O'][1], 
                p0_tau=0.50, fixed_cen=3.00
            )
            
            t_co2, c_co2, w_co2, err_co2_tau = fit_single_ice(
                tau_raw, waves, fit_ranges['CO2'][0], fit_ranges['CO2'][1], 
                p0_tau=0.20, p0_wid=0.04, fixed_cen=4.25, wid_bounds=(0.02, 0.08)
            )
            
            # RESTRICTED CO SIGMA FIT (Narrower width bounds and guess)
            t_co, c_co, w_co, err_co_tau = fit_single_ice(
                tau_raw, waves, fit_ranges['CO'][0], fit_ranges['CO'][1], 
                p0_tau=0.10, p0_wid=0.02, fixed_cen=4.65, wid_bounds=(0.005, 0.03)
            )

            maps_tau_peak['H2O'][y, x] = t_h2o
            maps_tau_peak['CO2'][y, x] = t_co2
            maps_tau_peak['CO'][y, x] = t_co

            # INTEGRATION & MAPPING
            if a_pah > 0:
                maps['PAH_3.3'][y, x] = np.trapz(pah_33_array[sort_idx], x=nu_sorted)
            if a_34 > 0:
                maps['PAH_3.4'][y, x] = np.trapz(pah_34_array[sort_idx], x=nu_sorted)
            
            h2o_array = asymmetric_gaussian(waves, t_h2o, c_h2o, wl_h2o, wr_h2o)
            if t_h2o > 0:
                maps['H2O'][y, x] = np.trapz(h2o_array[sort_idx], x=nu_sorted) / ices['H2O'][2]
                maps_err['H2O'][y, x] = (err_h2o_tau / t_h2o) * maps['H2O'][y, x]
            
            co2_array = gaussian(waves, t_co2, c_co2, w_co2)
            if t_co2 > 0:
                maps['CO2'][y, x] = np.trapz(co2_array[sort_idx], x=nu_sorted) / ices['CO2'][2]
                maps_err['CO2'][y, x] = (err_co2_tau / t_co2) * maps['CO2'][y, x]
            
            co_array = gaussian(waves, t_co, c_co, w_co)
            if t_co > 0:
                maps['CO'][y, x] = np.trapz(co_array[sort_idx], x=nu_sorted) / ices['CO'][2]
                maps_err['CO'][y, x] = (err_co_tau / t_co) * maps['CO'][y, x]

            tau_corr_cube[:, y, x] = h2o_array + co2_array + co_array
            
            pixel_cont_150 = np.interp(1.50, waves, cont_spec)
            if med_cont_150 > 0 and pixel_cont_150 > 0:
                maps['Flux_Ratio'][y, x] = pixel_cont_150 / med_cont_150 

        if (y + 1) % 10 == 0:
            print(f"    Row {y + 1}/{ny} complete")

    # ==========================================
    # SECOND-PASS: DIAGNOSTIC PLOTTING
    # ==========================================
    h2o_plot_threshold = tau_thresholds.get('H2O', 0.05)
    valid_y, valid_x = np.where(maps_tau_peak['H2O'] > h2o_plot_threshold)
    num_qualified = len(valid_y)
    
    with PdfPages(output_diag) as pdf:
        if num_qualified > 0:
            num_to_plot = min(num_qualified, 100)
            chosen_indices = np.random.choice(num_qualified, size=num_to_plot, replace=False)
            
            for idx in chosen_indices:
                y, x = valid_y[idx], valid_x[idx]
                flux = data[:, y, x]
                cont_spec = continuum_cube[:, y, x]
                tau_raw = tau_raw_cube[:, y, x]
                
                t_h2o = maps_tau_peak['H2O'][y, x]
                t_co2 = maps_tau_peak['CO2'][y, x]
                t_co  = maps_tau_peak['CO'][y, x]
                
                _, _, wl_h2o, wr_h2o, _ = fit_h2o_asymmetric(tau_raw, waves, fit_ranges['H2O'][0], fit_ranges['H2O'][1], 0.5, 3.0)
                _, _, w_co2, _ = fit_single_ice(tau_raw, waves, fit_ranges['CO2'][0], fit_ranges['CO2'][1], 0.2, 0.04, 4.25, (0.02, 0.08))
                # RESTRICTED CO SIGMA FIT
                _, _, w_co, _ = fit_single_ice(tau_raw, waves, fit_ranges['CO'][0], fit_ranges['CO'][1], 0.1, 0.02, 4.65, (0.005, 0.03))
                
                h2o_array = asymmetric_gaussian(waves, t_h2o, 3.00, wl_h2o, wr_h2o)
                co2_array = gaussian(waves, t_co2, 4.25, w_co2)
                co_array  = gaussian(waves, t_co, 4.65, w_co)
                
                a_pah, c_pah, w_pah, a_34, c_34, w_34 = evaluate_and_fit_pah_features(
                    waves, flux, subtract_pah=subtract_pah, fit_34_feature=fit_34_feature, 
                    thresh_33=thresh_33, thresh_34=thresh_34
                )

                pah_33_array = gaussian(waves, a_pah, c_pah, w_pah) if a_pah > 0 else np.zeros_like(waves)
                pah_34_array = gaussian(waves, a_34, c_34, w_34) if a_34 > 0 else np.zeros_like(waves)
                total_pah = pah_33_array + pah_34_array

                # DIAGNOSTIC FIGURE
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
                
                for ax in (ax1, ax2):
                    ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, labelsize=10)
                    ax.grid(True, linestyle=':', alpha=0.45)

                ax1.plot(waves, flux, label='Observed Flux', color='black', drawstyle='steps-mid', linewidth=1)
                ax1.plot(waves, cont_spec, label='PCHIP Continuum', color='crimson', linestyle='--', linewidth=1.5)
                
                total_tau_fit = h2o_array + co2_array + co_array
                full_model = cont_spec * np.exp(-total_tau_fit) + total_pah
                ax1.plot(waves, full_model, label='Full Combined Fit', color='royalblue', alpha=0.85, linewidth=1.5)
                
                if a_pah > 0:
                    ax1.fill_between(waves, cont_spec, cont_spec + pah_33_array, color='forestgreen', alpha=0.3, label=r'PAH $3.3\,\mu\mathrm{m}$')
                if a_34 > 0:
                    ax1.fill_between(waves, cont_spec, cont_spec + pah_34_array, color='mediumseagreen', alpha=0.3, label=r'PAH/Aliphatic $3.4\,\mu\mathrm{m}$')

                ax1.set_ylabel(f'Flux Density [{flux_unit}]', fontsize=11)
                ax1.set_title(f'Diagnostic Fit — Pixel (X={x}, Y={y}) | Peak $\\tau(\\mathrm{{H_2O}})={t_h2o:.2f}$', fontsize=11, pad=8)
                
                valid_f = flux[np.isfinite(flux) & (flux > 0)]
                if len(valid_f) > 0:
                    ax1.set_ylim(max(0, np.min(valid_f) * 0.8), np.max(valid_f) * 1.38)

                ax1.legend(loc='upper left', ncol=2, frameon=True, framealpha=0.9, edgecolor='lightgray', fontsize=8.5)

                ax2.plot(waves, tau_raw, label=r'Raw Optical Depth ($\tau$)', color='black', drawstyle='steps-mid', linewidth=1)
                ax2.axvspan(fit_ranges['H2O'][0], fit_ranges['H2O'][1], color='cyan', alpha=0.08)
                ax2.axvspan(fit_ranges['CO2'][0], fit_ranges['CO2'][1], color='purple', alpha=0.08)
                ax2.axvspan(fit_ranges['CO'][0], fit_ranges['CO'][1], color='orange', alpha=0.08)
                
                ax2.plot(waves, h2o_array, label=r'$\mathrm{H_2O}$ Ice Fit', color='darkcyan', linewidth=1.8)
                ax2.plot(waves, co2_array, label=r'$\mathrm{CO_2}$ Ice Fit', color='purple', linewidth=1.8)
                ax2.plot(waves, co_array, label=r'$\mathrm{CO}$ Ice Fit', color='darkorange', linewidth=1.8)
                ax2.plot(waves, total_tau_fit, label=r'Total $\tau$ Model', color='crimson', linestyle='--', linewidth=1.8)
                
                ax2.set_ylabel(r'Optical Depth ($\tau$)', fontsize=11)
                ax2.set_xlabel(r'Wavelength ($\mu\mathrm{m}$)', fontsize=11)
                
                valid_tau_mask = (waves < 5.0) & ~np.isnan(tau_raw)
                if np.sum(valid_tau_mask) > 0:
                    y_max = min(np.nanmax(tau_raw[valid_tau_mask]) * 1.30, 5.0)
                    ax2.set_ylim(-0.05, max(y_max, 0.85))
                
                ax2.legend(loc='upper right', ncol=3, frameon=True, framealpha=0.9, edgecolor='lightgray', fontsize=8.5)
                
                plt.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)

    # ==========================================
    # MASKING & OUTPUT GENERATION
    # ==========================================
    h2o_thresh = tau_thresholds.get('H2O')
    co2_thresh = tau_thresholds.get('CO2')
    co_thresh = tau_thresholds.get('CO')
    
    if h2o_thresh is not None:
        mask_h2o = maps_tau_peak['H2O'] > h2o_thresh
        maps['H2O'] = np.where(mask_h2o, maps['H2O'], np.nan)
        maps_err['H2O'] = np.where(mask_h2o, maps_err['H2O'], np.nan)
        maps['PAH_3.3'] = np.where(mask_h2o, maps['PAH_3.3'], np.nan)
        maps['PAH_3.4'] = np.where(mask_h2o, maps['PAH_3.4'], np.nan)
        
    if co2_thresh is not None:
        mask_co2 = maps_tau_peak['CO2'] > co2_thresh
        maps['CO2'] = np.where(mask_co2, maps['CO2'], np.nan)
        maps_err['CO2'] = np.where(mask_co2, maps_err['CO2'], np.nan)
        
    if co_thresh is not None:
        mask_co = maps_tau_peak['CO'] > co_thresh
        maps['CO'] = np.where(mask_co, maps['CO'], np.nan)
        maps_err['CO'] = np.where(mask_co, maps_err['CO'], np.nan)

    print(f"\nSaving FITS to {output_fits}...")
    try:
        wcs_2d = WCS(header).celestial
        if wcs_2d.naxis < 2: wcs_2d = None
    except Exception:
        wcs_2d = None
        
    header_2d = wcs_2d.to_header() if wcs_2d else header.copy()
    
    hdus = [fits.PrimaryHDU(header=header_2d)]
    for key, map_data in maps.items():
        hdu = fits.ImageHDU(data=map_data, header=header_2d, name=f"{key}_MAP")
        if 'PAH' in key:
            hdu.header['BUNIT'] = 'cm-1'
        elif key == 'Flux_Ratio':
            hdu.header['BUNIT'] = 'Ratio'
        else:
            hdu.header['BUNIT'] = 'molecules cm-2'
        hdus.append(hdu)
        
        if key in maps_err:
            hdu_err = fits.ImageHDU(data=maps_err[key], header=header_2d, name=f"{key}_ERR_MAP")
            hdu_err.header['BUNIT'] = 'molecules cm-2'
            hdus.append(hdu_err)
            
    hdus.extend([
        fits.ImageHDU(data=continuum_cube, header=header, name="CONTINUUM_FIT"),
        fits.ImageHDU(data=tau_raw_cube, header=header, name="TAU_RAW"),
        fits.ImageHDU(data=tau_corr_cube, header=header, name="TAU_FITTED_MODEL")
    ])
    fits.HDUList(hdus).writeto(output_fits, overwrite=True)
    
    print("Plotting individual maps to PNG...")
    plot_params = {
        'H2O': (r'$\mathrm{H_2O}$ Ice Column Density', 'Blues', r'$\mathrm{molecules\ cm^{-2}}$', AsymmetricPercentileInterval(1, 99), LinearStretch()),
        'CO2': (r'$\mathrm{CO_2}$ Ice Column Density', 'Purples', r'$\mathrm{molecules\ cm^{-2}}$', AsymmetricPercentileInterval(1, 99), LinearStretch()),
        'CO':  (r'$\mathrm{CO}$ Ice Column Density',  'Oranges', r'$\mathrm{molecules\ cm^{-2}}$', AsymmetricPercentileInterval(1, 99), LinearStretch()),
        'PAH_3.3': (r'$3.3\,\mu\mathrm{m}$ PAH Integrated Area', 'Greens', r'$\mathrm{cm^{-1}}$', AsymmetricPercentileInterval(1, 99), LinearStretch()),
        'PAH_3.4': (r'$3.4\,\mu\mathrm{m}$ Feature Integrated Area', 'YlGn', r'$\mathrm{cm^{-1}}$', AsymmetricPercentileInterval(1, 99), LinearStretch()),
        'Flux_Ratio': (r'Continuum Ratio ($1.50\,\mu\mathrm{m}$ / Median)', 'inferno', 'Ratio', AsymmetricPercentileInterval(1, 90), LinearStretch())
    }
    
    for key in maps.keys():
        title, cmap, unit, interval, stretch = plot_params[key]
        map_data = maps[key]
        
        fig, ax = plt.subplots(figsize=(7, 6), subplot_kw={'projection': wcs_2d} if wcs_2d else {})
        valid_values = map_data[np.isfinite(map_data)]
        
        if len(valid_values) > 0:
            vmin, vmax = interval.get_limits(valid_values)
            vmin = min(0, vmin)
            norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=stretch)
            
            cmap_obj = plt.get_cmap(cmap).copy()
            cmap_obj.set_bad(cmap_obj(0))
            
            plot_data = np.nan_to_num(map_data, nan=0.0)
            im = ax.imshow(plot_data, origin='lower', cmap=cmap_obj, interpolation='nearest', norm=norm)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(unit)
        else:
            ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', transform=ax.transAxes)
            
        ax.set_title(title, fontsize=13)
        ax.set_xlabel('Right Ascension' if wcs_2d else 'Pixel X')
        ax.set_ylabel('Declination' if wcs_2d else 'Pixel Y')
            
        plt.tight_layout()
        plt.savefig(f"{output_png_base}_{key}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)

    print("Saving to CSV...")
    y_coords, x_coords = np.mgrid[0:ny, 0:nx]
    y_flat, x_flat = y_coords.flatten(), x_coords.flatten()
    
    ra_flat, dec_flat = (wcs_2d.pixel_to_world_values(x_flat, y_flat) if wcs_2d 
                         else (np.full_like(x_flat, np.nan, dtype=float), np.full_like(y_flat, np.nan, dtype=float)))
        
    df_out = pd.DataFrame({
        'x': x_flat, 'y': y_flat, 'ra': ra_flat, 'dec': dec_flat,
        'N_H2O': maps['H2O'].flatten(), 'N_H2O_err': maps_err['H2O'].flatten(),
        'N_CO2': maps['CO2'].flatten(), 'N_CO2_err': maps_err['CO2'].flatten(),
        'N_CO': maps['CO'].flatten(), 'N_CO_err': maps_err['CO'].flatten(),
        'Area_PAH_33': maps['PAH_3.3'].flatten(), 'Area_PAH_34': maps['PAH_3.4'].flatten()
    }).dropna(subset=['N_H2O'])
    
    df_out.to_csv(output_fits.replace('.fits', '.csv'), index=False)
    print("Done!")




In [ ]:
if __name__ == "__main__":
    cube_file = "spherex_cube_V347 Aur_epoch2.fits" 
    target_name = cube_file.replace("spherex_cube_", "").replace(".fits", "")
    
    process_datacube(
        cube_path=cube_file, 
        output_fits=f"ice_maps_{target_name}.fits", 
        output_png_base=f"ice_map_{target_name}",
        output_diag=f"continuum_diagnostics_{target_name}.pdf", 
        subtract_pah=True, 
        fit_34_feature=True,
        thresh_33=3.0,  # Signal-to-noise threshold to trigger 3.3 um fit
        thresh_34=3.0,  # Signal-to-noise threshold to trigger 3.4 um fit
        bin_factor=1, 
        tau_thresholds={
            'H2O': 0.4,
            'CO2': 0.25,
            'CO': 0.25
        }
    )

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter
from scipy import odr
from scipy.stats import gaussian_kde


def fit_and_plot_correlation(ax, x_val, y_val, x_err, y_err, xlabel, ylabel, title, cmap, thresh_x, thresh_y, min_snr=10.0):
    """
    Filters data at >10-sigma, performs scaled Orthogonal Distance Regression (ODR) 
    weighted by spaxel cluster density for a strict linear model (y = m*x + c), 
    and overlays the trendline.
    """

    # 1. Filter: Above thresholds AND fit significance > 10 sigma (S/N >= 10)
    snr_x = x_val / x_err
    snr_y = y_val / y_err

    mask = (
        (x_val > thresh_x) &
        (y_val > thresh_y) &
        (snr_x >= min_snr) &
        (snr_y >= min_snr) &
        np.isfinite(x_val) &
        np.isfinite(y_val) &
        np.isfinite(x_err) &
        np.isfinite(y_err) &
        (x_err > 0) &
        (y_err > 0)
    )

    xv = x_val[mask].values
    yv = y_val[mask].values
    xev = x_err[mask].values
    yev = y_err[mask].values

    if len(xv) > 2:
        # 2. Compute 2D Kernel Density Estimation (KDE) to isolate the cluster core
        log_x = np.log10(xv)
        log_y = np.log10(yv)
        xy_data = np.vstack([log_x, log_y])
        kde = gaussian_kde(xy_data)
        density = kde(xy_data)
        density_norm = (density - density.min()) / (density.max() - density.min() + 1e-12)

        # 3. Scale variables to ~1.0 for numerical precision during ODR fitting
        x_scale = 1e18
        y_scale = 1e18

        xs = xv / x_scale
        ys = yv / y_scale
        xes = xev / x_scale
        yes = yev / y_scale

        # Weighting: density^2 heavily prioritizes the dense spaxel core
        weight_factor = (density_norm ** 2) + 0.01
        sx = xes / np.sqrt(weight_factor)
        sy = yes / np.sqrt(weight_factor)

        # 4. Orthogonal Distance Regression (ODR) for y = m*x + c (handles both x and y errors)
        def f_lin(p, x):
            return p[0] * x + p[1]

        linear_model = odr.Model(f_lin)
        data = odr.RealData(xs, ys, sx=sx, sy=sy)
        
        # Initial guess using scaled means
        m_init = np.mean(ys) / np.mean(xs)
        odr_run = odr.ODR(data, linear_model, beta0=[m_init, 0.0]).run()

        # Extract scaled parameters
        m_scaled, c_scaled = odr_run.beta
        m_err_scaled, c_err_scaled = odr_run.sd_beta

        # 5. Convert parameters back to original unscaled units
        m = m_scaled * (y_scale / x_scale)
        c = c_scaled * y_scale
        m_err = m_err_scaled * (y_scale / x_scale)
        c_err = c_err_scaled * y_scale

        # R-squared calculation
        y_pred = m * xv + c
        ss_res = np.sum((yv - y_pred)**2)
        ss_tot = np.sum((yv - np.mean(yv))**2)
        r_squared = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

        # Generate linear trendline domain
        x_line = np.logspace(np.log10(xv.min() * 0.75), np.log10(xv.max() * 1.35), 300)
        y_line = m * x_line + c

        # Filter out non-positive values for log-log rendering
        valid_line = y_line > 0
        x_line_plot = x_line[valid_line]
        y_line_plot = y_line[valid_line]

        # -----------------------------------------------------------------
        # VISUALIZATION
        # -----------------------------------------------------------------
        # Errorbars behind markers
        ax.errorbar(
            xv, yv, xerr=xev, yerr=yev,
            fmt='none', ecolor='#808080', elinewidth=1.0, capsize=0,
            alpha=0.35, zorder=2
        )

        # Density Scatter Plot
        sc = ax.scatter(
            xv, yv, c=density_norm, cmap=cmap, s=50,
            edgecolor='black', linewidth=0.6, alpha=0.9,
            zorder=3, label=f'Data ($>{int(min_snr)}\\sigma$)'
        )

        # Colorbar
        cbar = plt.colorbar(sc, ax=ax, pad=0.02, fraction=0.046)
        cbar.set_label('Spaxel Cluster Density (KDE)', fontsize=12)
        cbar.ax.tick_params(labelsize=10)

        # Linear Trendline (y = mx + c)
        ax.plot(
            x_line_plot, y_line_plot, color='black', linewidth=3.0,
            linestyle='--', zorder=4, label='Cluster ODR Fit'
        )

        # Format c in scientific notation for display
        sign_c = "+" if c >= 0 else "-"
        text_str = (
            f"$y = ({m:.3f} \\pm {m_err:.3f})x {sign_c} {abs(c):.2e}$\n"
            f"$R^2 = {r_squared:.2f}$  |  $N = {len(xv)}$ ($>{int(min_snr)}\\sigma$)"
        )
        props = dict(boxstyle='round,pad=0.6', facecolor='white', alpha=0.92, edgecolor='#A0A0A0', linewidth=1.2)
        ax.text(0.04, 0.94, text_str, transform=ax.transAxes, fontsize=12, fontweight='medium',
                verticalalignment='top', bbox=props, zorder=5)

        print("\n" + "="*60)
        print(f"{title} (Scaled Linear ODR Fit: y = m*x + c)")
        print(f"  Slope (m)     : {m:.5f} +/- {m_err:.5f}")
        print(f"  Intercept (c) : {c:.5e} +/- {c_err:.5e}")

        ax.set_xlim(xv.min() * 0.75, xv.max() * 1.35)
        ax.set_ylim(yv.min() * 0.75, yv.max() * 1.35)

    else:
        ax.text(
            0.5, 0.5, f'Insufficient data\n($>{int(min_snr)}\\sigma$ cut)',
            ha='center', va='center',
            transform=ax.transAxes, fontsize=18, fontweight='bold', color='gray'
        )

    # -----------------------------------------------------------------
    # AXES AND TYPOGRAPHY STYLING
    # -----------------------------------------------------------------
    ax.set_xlabel(xlabel, fontsize=16, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=16, labelpad=10)
    ax.set_title(title, fontsize=18, fontweight='bold', pad=14)

    #ax.set_xscale('log')
    #ax.set_yscale('log')

    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))

    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())

    ax.tick_params(which='major', direction='in', top=True, right=True, length=8, width=1.5, labelsize=14)
    ax.tick_params(which='minor', direction='in', top=True, right=True, length=4, width=1.0)

    ax.grid(which='major', linestyle='-', linewidth=0.8, alpha=0.25, color='gray')
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.15, color='gray')

    for spine in ax.spines.values():
        spine.set_linewidth(2.0)


def main(csv_filename, thresholds={'H2O': 0.1e17, 'CO2': 0.05e17, 'CO': 0.02e17}, min_snr=10.0):

    print(f"Loading data from {csv_filename}...")
    df = pd.read_csv(csv_filename)

    fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))

    # H2O vs CO2
    fit_and_plot_correlation(
        axes[0], df['N_H2O'], df['N_CO2'], df['N_H2O_err'], df['N_CO2_err'],
        r'$N(\mathrm{H_2O})\ [\mathrm{cm^{-2}}]$', r'$N(\mathrm{CO_2})\ [\mathrm{cm^{-2}}]$',
        r'$\mathrm{H_2O\ vs\ CO_2}$', 'magma', thresholds['H2O'], thresholds['CO2'],
        min_snr=min_snr
    )

    # H2O vs CO
    fit_and_plot_correlation(
        axes[1], df['N_H2O'], df['N_CO'], df['N_H2O_err'], df['N_CO_err'],
        r'$N(\mathrm{H_2O})\ [\mathrm{cm^{-2}}]$', r'$N(\mathrm{CO})\ [\mathrm{cm^{-2}}]$',
        r'$\mathrm{H_2O\ vs\ CO}$', 'viridis', thresholds['H2O'], thresholds['CO'],
        min_snr=min_snr
    )

    # CO2 vs CO
    fit_and_plot_correlation(
        axes[2], df['N_CO2'], df['N_CO'], df['N_CO2_err'], df['N_CO_err'],
        r'$N(\mathrm{CO_2})\ [\mathrm{cm^{-2}}]$', r'$N(\mathrm{CO})\ [\mathrm{cm^{-2}}]$',
        r'$\mathrm{CO_2\ vs\ CO}$', 'plasma', thresholds['CO2'], thresholds['CO'],
        min_snr=min_snr
    )

    plt.tight_layout()
    output_png = "scatter_fits_linear_odr.png"
    plt.savefig(output_png, dpi=400, bbox_inches='tight')
    print(f"\nSaved linear ODR fit plot to {output_png}")


if __name__ == "__main__":
    target_csv = "ice_maps_Serpens_NIRCam_epoch1.csv"
    main(
        target_csv,
        thresholds={
            'H2O': 2e18,
            'CO2': 1e17,
            'CO': 1e17
        },
        min_snr=5
    )

In [ ]:
pwd